# U-Net Glomeruli Segmentation — Binary Segmentation Training Pipeline

This notebook implements a complete PyTorch U-Net training system for binary semantic segmentation of glomeruli from renal biopsy whole-slide images.

## Two-Model Strategy

This pipeline uses a **two-stage approach** for glomeruli classification:

1. **Model 1 (this notebook)**: Binary segmentation to detect glomerulus locations
   - Input: Z-score normalized RGB tiles from WSI
   - Output: Pixel-level binary masks (background vs glomerulus)
   - Classes: 0 (background), 1 (glomerulus — fused from original classes 1-4)
   - Purpose: Localizes all glomeruli regardless of subtype

2. **Model 2 (future)**: Classifier on detected patches to assign classes 1-4
   - Input: Image patches extracted from detected glomerulus regions (Model 1)
   - Output: Per-glomerulus classification (classes 1-4: FGS, MPGN, FSGS, other)
   - Purpose: Fine-grained subtype classification for clinical diagnosis

## Notebook Contents

1. **Data Loading** — GlomeruliDataset with grouped split and online augmentation (train only)
2. **Loss Functions** — Dice Loss + CrossEntropy + Class Weighting
3. **U-Net Architecture** — Encoder-Decoder with Skip Connections
4. **Binary Segmentation Metrics** — Accuracy, Precision, Recall, F1, ROC-AUC, Dice
5. **Training Pipeline** — Full training loop with validation and checkpointing

## Pipeline Overview

WSI TIFF + GeoJSON annotations → Tiled images + masks (classes 0-4) →
Reinhard color normalization → Z-score standardization →
Train/Val/Test split (grouped by biopsy, 70/15/15) →
Online augmentation (train only) → U-Net training → Model evaluation (Binary Metrics)

See WORKFLOW.md for end-to-end pipeline documentation.

In [1]:
# Standard library
import os
import json
from pathlib import Path
from datetime import datetime
from typing import Tuple
import random

# Scientific computing
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR, LinearLR
from torch.cuda.amp import autocast, GradScaler
from torch.utils.tensorboard import SummaryWriter
import albumentations as A

# Data loading
import cv2
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset
import warnings

# Check device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

# --- CUDA performance tuning (T4 Tensor Cores) ---
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = True          # auto-tune conv algorithms
    torch.backends.cuda.matmul.allow_tf32 = True   # TF32 for matmul (T4 supports it)
    torch.backends.cudnn.allow_tf32 = True         # TF32 for cudnn convolutions
    print(f"cudnn.benchmark: {torch.backends.cudnn.benchmark}")
    print(f"CUDA matmul TF32: {torch.backends.cuda.matmul.allow_tf32}")
    print(f"cudnn TF32: {torch.backends.cudnn.allow_tf32}")
    print(f"PYTORCH_CUDA_ALLOC_CONF: expandable_segments:True")


# --- Preprocessing Transforms (Reinhard Normalization + Z-score) ---

class ReinhardNormalize:
    """Reinhard stain normalization in LAB color space for consistency across slides."""

    def __init__(self, target_stats: dict):
        self.target = target_stats

    @staticmethod
    def _get_tissue_mask(img_bgr: np.ndarray) -> np.ndarray:
        """Isolate tissue pixels from background and artifacts via luminance and saturation thresholds."""
        lab = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2LAB)
        hsv = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)
        mask = (lab[:, :, 0] < 230) & (hsv[:, :, 1] > 10)
        kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (9, 9))
        mask = cv2.morphologyEx(mask.astype(np.uint8), cv2.MORPH_CLOSE, kernel)
        kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
        mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
        return mask.astype(bool)

    @staticmethod
    def compute_template_stats(image_paths: list, n_samples: int = 200) -> dict:
        """Compute median LAB statistics from tissue pixels to define normalization target."""
        paths_list = list(image_paths)[:n_samples]
        sample = random.sample(paths_list, min(n_samples, len(paths_list)))
        all_stats = []
        
        for p in sample:
            img = cv2.imread(str(p))
            if img is None:
                continue
            tissue = ReinhardNormalize._get_tissue_mask(img)
            if tissue.sum() < 100:
                continue
            lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB).astype(np.float32)
            stats = ([lab[..., c][tissue].mean() for c in range(3)] +
                     [lab[..., c][tissue].std()  for c in range(3)])
            all_stats.append(stats)
        
        if not all_stats:
            return {'mean_L': 50, 'mean_a': 128, 'mean_b': 128,
                    'std_L': 10, 'std_a': 10, 'std_b': 10}
        
        arr = np.array(all_stats)
        keys = ['mean_L', 'mean_a', 'mean_b', 'std_L', 'std_a', 'std_b']
        return {k: float(np.median(arr[:, i])) for i, k in enumerate(keys)}

    def __call__(self, img_bgr: np.ndarray) -> np.ndarray:
        """Normalize image to match template statistics, preserving background pixels."""
        tissue = self._get_tissue_mask(img_bgr)
        if tissue.sum() < 100:
            return img_bgr
        
        lab = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2LAB).astype(np.float32)
        
        src = {c: (lab[..., i][tissue].mean(), lab[..., i][tissue].std())
               for i, c in enumerate(['L', 'a', 'b'])}
        
        result = lab.copy()
        for i, c in enumerate(['L', 'a', 'b']):
            m, s = src[c]
            result[..., i] = ((lab[..., i] - m) *
                              (self.target[f'std_{c}'] / (s + 1e-5)) +
                              self.target[f'mean_{c}'])
        
        result[~tissue] = lab[~tissue]
        result = np.clip(result, 0, 255).astype(np.uint8)
        return cv2.cvtColor(result, cv2.COLOR_LAB2BGR)


def compute_channel_stats(image_paths: list, n_samples: int = 200) -> Tuple[list, list]:
    """Compute weighted per-channel RGB stats from tissue pixels for Z-score normalization."""
    paths_list = list(image_paths)[:n_samples]
    sample = random.sample(paths_list, min(n_samples, len(paths_list)))
    
    tile_means, tile_vars, tile_counts = [], [], []
    for p in sample:
        img_bgr = cv2.imread(str(p))
        if img_bgr is None:
            continue
        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
        tissue = ReinhardNormalize._get_tissue_mask(img_bgr)
        if tissue.sum() < 100:
            continue
        
        pixels = img_rgb[tissue]
        tile_means.append(pixels.mean(0))
        tile_vars.append(pixels.var(0))
        tile_counts.append(tissue.sum())
    
    if not tile_means:
        return [0.5, 0.5, 0.5], [0.2, 0.2, 0.2]
    
    means_arr = np.array(tile_means)
    vars_arr = np.array(tile_vars)
    counts_arr = np.array(tile_counts, dtype=np.float64)
    w = counts_arr / counts_arr.sum()
    
    mean = (means_arr * w[:, None]).sum(0)
    var = ((vars_arr + (means_arr - mean) ** 2) * w[:, None]).sum(0)
    
    return mean.tolist(), np.sqrt(var).tolist()


Using device: cuda
GPU: Tesla T4
VRAM: 17.00 GB
cudnn.benchmark: True
CUDA matmul TF32: True
cudnn TF32: True
PYTORCH_CUDA_ALLOC_CONF: expandable_segments:True


In [2]:
def _collect_image_tiles(images_dir: str) -> list:
    """Collect only input tiles from */images/*.png, never generated masks."""
    images_dir = Path(images_dir)
    image_paths = sorted(images_dir.glob('*/images/*.png'))

    # Fallback for flat/custom datasets: include PNGs except anything inside a masks folder
    # or files already named *_mask.png.
    if not image_paths:
        image_paths = sorted(
            p for p in images_dir.rglob('*.png')
            if 'masks' not in p.relative_to(images_dir).parts
            and not p.stem.endswith('_mask')
        )

    return image_paths


def split_biopsias(
    images_dir: str,
    train_size: float = 0.70,
    val_size: float = 0.15,
    seed: int = 42,
) -> Tuple[list, list, list, dict]:
    """Groups biopsias into train/val/test to prevent data leakage at slide level."""
    images_dir = Path(images_dir)

    all_images = _collect_image_tiles(images_dir)

    if not all_images:
        raise ValueError(f"No PNG image tiles found in {images_dir}. Expected files under */images/*.png")

    biopsias_dict = {}
    for img_path in all_images:
        relative = img_path.relative_to(images_dir)
        biopsia = relative.parts[0]
        if biopsia not in biopsias_dict:
            biopsias_dict[biopsia] = []
        biopsias_dict[biopsia].append(img_path)

    biopsias_list = list(biopsias_dict.keys())

    test_size = 1.0 - train_size - val_size
    assert test_size >= 0, "train_size + val_size must be <= 1.0"

    train_val_biopsias, test_biopsias = train_test_split(
        biopsias_list,
        test_size=test_size if test_size > 0 else None,
        random_state=seed,
    )

    if val_size > 0:
        val_fraction = val_size / (train_size + val_size)
        train_biopsias, val_biopsias = train_test_split(
            train_val_biopsias,
            test_size=val_fraction,
            random_state=seed + 1,
        )
    else:
        train_biopsias = train_val_biopsias
        val_biopsias = []

    return train_biopsias, val_biopsias, test_biopsias, biopsias_dict



In [3]:
class GlomeruliDataset(Dataset):
    """Loads paired image-mask glomeruli tiles with online preprocessing and augmentation."""

    def __init__(
        self,
        images_dir: str,
        masks_dir: str = None,
        split: str = 'train',
        biopsias: list = None,
        biopsias_dict: dict = None,
        reinhard_norm=None,
        channel_means: list = None,
        channel_stds: list = None,
        train_size: float = 0.70,
        val_size: float = 0.15,
        seed: int = 42,
        transforms=None,
    ):
        self.images_dir = Path(images_dir)
        self.masks_dir = Path(masks_dir) if masks_dir is not None else Path(images_dir)
        self.split = split
        self.transforms = transforms
        self.reinhard_norm = reinhard_norm
        self.channel_means = channel_means if channel_means is not None else [0.5, 0.5, 0.5]
        self.channel_stds = channel_stds if channel_stds is not None else [0.2, 0.2, 0.2]

        assert split in {'train', 'val', 'test'}, f"Invalid split: {split}"
        assert self.images_dir.exists(), f"Images dir not found: {self.images_dir}"
        assert self.masks_dir.exists(), f"Masks dir not found: {self.masks_dir}"

        if biopsias is not None and biopsias_dict is not None:
            selected_biopsias = biopsias
            self.biopsias_dict = biopsias_dict
        else:
            test_size = 1.0 - train_size - val_size
            assert test_size >= 0, "train_size + val_size must be <= 1.0"

            all_images = _collect_image_tiles(self.images_dir)
            if not all_images:
                raise ValueError(f"No PNG image tiles found in {self.images_dir}. Expected files under */images/*.png")

            biopsias_dict = {}
            for img_path in all_images:
                relative = img_path.relative_to(self.images_dir)
                biopsia = relative.parts[0]
                if biopsia not in biopsias_dict:
                    biopsias_dict[biopsia] = []
                biopsias_dict[biopsia].append(img_path)

            self.biopsias_dict = biopsias_dict
            biopsias_list = list(biopsias_dict.keys())

            train_val_biopsias, test_biopsias = train_test_split(
                biopsias_list,
                test_size=test_size if test_size > 0 else None,
                random_state=seed,
            )

            if val_size > 0:
                val_fraction = val_size / (train_size + val_size)
                train_biopsias, val_biopsias = train_test_split(
                    train_val_biopsias,
                    test_size=val_fraction,
                    random_state=seed + 1,
                )
            else:
                train_biopsias = train_val_biopsias
                val_biopsias = []

            if split == 'train':
                selected_biopsias = train_biopsias
            elif split == 'val':
                selected_biopsias = val_biopsias
            else:
                selected_biopsias = test_biopsias

        self.image_paths = []
        for biopsia in selected_biopsias:
            self.image_paths.extend(self.biopsias_dict[biopsia])

        self.image_paths = sorted(self.image_paths)

        paired = []
        missing = []
        for img_path in self.image_paths:
            mask_path = self._get_mask_path(img_path)
            if mask_path.exists():
                paired.append((img_path, mask_path))
            else:
                missing.append((img_path, mask_path))

        if missing:
            warnings.warn(
                f"Found {len(missing)} images without corresponding masks. "
                f"These will be skipped. First few: {missing[:3]}"
            )

        if not paired:
            raise ValueError("No valid image-mask pairs found after checking.")

        self.image_paths, self.mask_paths = zip(*paired)
        self.image_paths = list(self.image_paths)
        self.mask_paths = list(self.mask_paths)

    def _get_mask_path(self, image_path: Path) -> Path:
        """Convert */images/<tile>.png to */masks/<tile>_mask.png."""
        rel = image_path.relative_to(self.images_dir)
        parts = list(rel.parts)

        if len(parts) < 3 or parts[1] != 'images':
            raise ValueError(
                f"Unexpected image path layout: {image_path}. "
                "Expected <root>/<biopsia>/images/<tile>.png"
            )

        parts[1] = 'masks'
        stem = Path(parts[-1]).stem
        parts[-1] = f"{stem}_mask.png"
        return self.masks_dir / Path(*parts)

    def __len__(self) -> int:
        return len(self.image_paths)

    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, torch.Tensor]:
        """Load BGR tile -> Reinhard normalization -> BGR-to-RGB -> Z-score -> augment -> tensors."""
        img_path = self.image_paths[idx]
        mask_path = self.mask_paths[idx]

        img_bgr = cv2.imread(str(img_path), cv2.IMREAD_COLOR)
        if img_bgr is None:
            raise RuntimeError(f"Failed to load image: {img_path}")

        if self.reinhard_norm is not None:
            img_bgr = self.reinhard_norm(img_bgr)

        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0

        if self.channel_means is not None and self.channel_stds is not None:
            for c in range(3):
                img_rgb[..., c] = (img_rgb[..., c] - self.channel_means[c]) / (self.channel_stds[c] + 1e-6)

        mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
        if mask is None:
            raise RuntimeError(f"Failed to load mask: {mask_path}")

        mask_binary = (mask > 0).astype(np.uint8)

        if self.transforms is not None and self.split == 'train':
            augmented = self.transforms(image=img_rgb, mask=mask_binary)
            img_rgb = augmented['image']
            mask_binary = augmented['mask']

        img_tensor = torch.from_numpy(np.transpose(img_rgb, (2, 0, 1))).float()
        mask_tensor = torch.from_numpy(mask_binary.astype(np.int64)).long()

        return img_tensor, mask_tensor


In [4]:
def create_dataloaders(
    images_dir: str,
    masks_dir: str = None,
    batch_size: int = 4,
    num_workers: int = 4,
    seed: int = 42,
    train_transforms=None,
    val_transforms=None,
    reinhard_norm=None,
    channel_means: list = None,
    channel_stds: list = None,
):
    """Create train/val/test DataLoaders with pre-computed grouped splits by biopsia."""
    train_biopsias, val_biopsias, test_biopsias, biopsias_dict = split_biopsias(
        images_dir=images_dir,
        train_size=0.70,
        val_size=0.15,
        seed=seed,
    )

    train_ds = GlomeruliDataset(
        images_dir,
        masks_dir,
        split='train',
        biopsias=train_biopsias,
        biopsias_dict=biopsias_dict,
        reinhard_norm=reinhard_norm,
        channel_means=channel_means,
        channel_stds=channel_stds,
        transforms=train_transforms,
    )

    val_ds = GlomeruliDataset(
        images_dir,
        masks_dir,
        split='val',
        biopsias=val_biopsias,
        biopsias_dict=biopsias_dict,
        reinhard_norm=reinhard_norm,
        channel_means=channel_means,
        channel_stds=channel_stds,
        transforms=val_transforms,
    )

    test_ds = GlomeruliDataset(
        images_dir,
        masks_dir,
        split='test',
        biopsias=test_biopsias,
        biopsias_dict=biopsias_dict,
        reinhard_norm=reinhard_norm,
        channel_means=channel_means,
        channel_stds=channel_stds,
        transforms=None,
    )

    train_loader = torch.utils.data.DataLoader(
        train_ds,
        batch_size=batch_size,
        shuffle=True,
        num_workers=num_workers,
        pin_memory=True,
        persistent_workers=(num_workers > 0),
        prefetch_factor=3 if num_workers > 0 else None,
    )

    val_loader = torch.utils.data.DataLoader(
        val_ds,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=True,
        persistent_workers=(num_workers > 0),
        prefetch_factor=2 if num_workers > 0 else None,
    )

    test_loader = torch.utils.data.DataLoader(
        test_ds,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=True,
        persistent_workers=(num_workers > 0),
        prefetch_factor=2 if num_workers > 0 else None,
    )

    return train_loader, val_loader, test_loader

In [5]:
# Quick test of dataset loading
if Path('Salidas/Tiles_UNet').exists():
    ds_test = GlomeruliDataset('Salidas/Tiles_UNet', split='train')
    print(f"Dataset loaded: {len(ds_test)} tiles")
    print(f"First image path: {ds_test.image_paths[0]}")
    print(f"First mask path:  {ds_test.mask_paths[0]}")

    img, mask = ds_test[0]
    print(f"Image shape: {img.shape}, dtype: {img.dtype}")
    print(f"Mask shape: {mask.shape}, dtype: {mask.dtype}")
    print(f"Mask unique classes (binary): {torch.unique(mask).tolist()}")
    print(f"Class distribution: 0 (background)={torch.sum(mask == 0).item()}, 1 (glomerulus)={torch.sum(mask == 1).item()}")

    positive_idx = next((i for i, p in enumerate(ds_test.mask_paths) if cv2.imread(str(p), cv2.IMREAD_GRAYSCALE).max() > 0), None)
    if positive_idx is not None:
        _, positive_mask = ds_test[positive_idx]
        print(f"Positive mask sanity check at idx={positive_idx}: classes={torch.unique(positive_mask).tolist()}, "
              f"glomerulus_pixels={torch.sum(positive_mask == 1).item()}")
    else:
        warnings.warn("No positive masks found in this split. Check tiling annotations or split selection.")
else:
    print("Dataset directory not found. Run preprocessing pipeline first.")
    print("Required: tiling_unet.py -> normalizacion.py")



Dataset loaded: 40360 tiles
First image path: Salidas\Tiles_UNet\18-139\images\18-139_tile_x05632_y12288_endx06656_endy13312.png
First mask path:  Salidas\Tiles_UNet\18-139\masks\18-139_tile_x05632_y12288_endx06656_endy13312_mask.png
Image shape: torch.Size([3, 1024, 1024]), dtype: torch.float32
Mask shape: torch.Size([1024, 1024]), dtype: torch.int64
Mask unique classes (binary): [0]
Class distribution: 0 (background)=1048576, 1 (glomerulus)=0
Positive mask sanity check at idx=30: classes=[0, 1], glomerulus_pixels=70323


## Part 2: Loss Functions for Class Imbalance

Semantic segmentation of medical images has extreme class imbalance: most pixels are background.

### Loss Strategy: CrossEntropy + Dice

- **CrossEntropy Loss**: Per-pixel multi-class classification (computed via `nn.CrossEntropyLoss`)
- **Dice Loss**: Penalizes lack of geometric overlap (Dice coefficient = 2|X∩Y|/(|X|+|Y|))
- **Class Weighting**: Inverse frequency weights (rare glomerulus class weighted higher)
- **Combined**: 50/50 weighted sum

In [6]:
class DiceLoss(nn.Module):
    """Sørensen-Dice coefficient for binary segmentation with per-class averaging."""

    def __init__(self, num_classes: int, smooth: float = 1e-6, ignore_background: bool = False):
        super().__init__()
        self.num_classes = num_classes
        self.smooth = smooth
        self.ignore_background = ignore_background

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        """Compute 1 - (mean Dice across classes)."""
        probs = torch.nn.functional.softmax(logits, dim=1)

        targets_one_hot = torch.nn.functional.one_hot(targets, num_classes=self.num_classes)
        targets_one_hot = targets_one_hot.permute(0, 3, 1, 2).float()

        dice_scores = []
        for c in range(self.num_classes):
            if self.ignore_background and c == 0:
                continue

            pred_c = probs[:, c, :, :]
            target_c = targets_one_hot[:, c, :, :]

            intersection = (pred_c * target_c).sum()
            union = pred_c.sum() + target_c.sum()

            dice = (2.0 * intersection + self.smooth) / (union + self.smooth)
            dice_scores.append(dice)

        if dice_scores:
            mean_dice = torch.stack(dice_scores).mean()
        else:
            mean_dice = torch.tensor(0.0, device=logits.device)

        return 1.0 - mean_dice

In [7]:
class CombinedLoss(nn.Module):
    """Weighted sum of CrossEntropy and Dice losses for class imbalance in segmentation."""

    def __init__(
        self,
        num_classes: int = 5,
        weight_ce: float = 0.5,
        weight_dice: float = 0.5,
        class_weights: torch.Tensor = None,
        smooth: float = 1e-6,
        ignore_background: bool = False,
    ):
        super().__init__()
        self.num_classes = num_classes
        self.weight_ce = weight_ce
        self.weight_dice = weight_dice
        self.smooth = smooth

        if class_weights is not None:
            self.register_buffer('class_weights', class_weights.float())
        else:
            self.register_buffer('class_weights', torch.ones(num_classes, dtype=torch.float32))

        self.dice_loss = DiceLoss(
            num_classes=num_classes,
            smooth=smooth,
            ignore_background=ignore_background,
        )

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        """Return weighted_ce * ce + weighted_dice * dice."""
        weights = self.class_weights.to(device=logits.device, dtype=logits.dtype)
        ce = torch.nn.functional.cross_entropy(logits, targets, weight=weights)
        dice = self.dice_loss(logits, targets)
        loss = self.weight_ce * ce + self.weight_dice * dice
        return loss


In [8]:
def compute_class_weights(mask_paths: list, num_classes: int = 2) -> torch.Tensor:
    """Compute inverse-frequency weights from train masks to handle extreme class imbalance."""
    class_counts = np.zeros(num_classes)

    for mask_path in mask_paths:
        mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
        if mask is None:
            continue
        mask_binary = (mask > 0).astype(np.uint8)
        for c in range(num_classes):
            class_counts[c] += (mask_binary == c).sum()

    class_counts = np.maximum(class_counts, 1.0)

    weights = 1.0 / class_counts
    weights = weights / weights.sum() * num_classes

    return torch.from_numpy(weights).float()

In [9]:
# Smoke test for loss functions with binary classes
B, C, H, W = 2, 2, 256, 256  # 2 classes for binary segmentation
logits = torch.randn(B, C, H, W, device=device, requires_grad=True)
targets = torch.randint(0, C, (B, H, W), device=device)

# Test DiceLoss
dice_loss = DiceLoss(num_classes=C)
dice = dice_loss(logits, targets)
print(f"Dice Loss: {dice.item():.4f}")

# Test CombinedLoss
combined = CombinedLoss(num_classes=C, weight_ce=0.5, weight_dice=0.5)
loss = combined(logits, targets)
print(f"Combined Loss: {loss.item():.4f}")

# Backward should work
loss.backward()
print("✓ Backward pass OK")


Dice Loss: 0.5011
Combined Loss: 0.7034
✓ Backward pass OK


## Part 3: U-Net Architecture

U-Net is a fully convolutional network with an encoder-decoder structure and skip connections.

### Architecture Overview

```
Input [B, 3, 1024, 1024]
    |
    v
Encoder (4 levels)
    |- Level 1: Conv->BN->ReLU x2, then MaxPool
    |- Level 2: Conv->BN->ReLU x2, then MaxPool
    |- Level 3: Conv->BN->ReLU x2, then MaxPool
    +- Level 4: Conv->BN->ReLU x2, then MaxPool
    |
    v
Bottleneck (highest compression, full context)
    |
    v
Decoder (4 levels, with skip connections from encoder)
    |- Level 1: Upsample + concat skip + Conv->BN->ReLU x2
    |- Level 2: Upsample + concat skip + Conv->BN->ReLU x2
    |- Level 3: Upsample + concat skip + Conv->BN->ReLU x2
    +- Level 4: Upsample + concat skip + Conv->BN->ReLU x2
    |
    v
Output [B, 2, 1024, 1024] (logits: 0=background, 1=glomerulus)
```

### Why Skip Connections?

Skip connections preserve fine-grained spatial information from the encoder in the decoder, enabling pixel-level precision.

### Binary Output Head

The final Conv2d outputs 2 channels (background vs glomerulus) instead of 5 multi-class outputs. This simplification allows the model to focus on glomerulus detection independent of subtype classification, which is handled by Model 2.

In [10]:
class DoubleConv(nn.Module):
    """Two consecutive Conv2d->BN->ReLU blocks."""

    def __init__(self, in_channels: int, out_channels: int, dropout: float = 0.0):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout) if dropout > 0 else nn.Identity(),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout) if dropout > 0 else nn.Identity(),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)

In [11]:
class Down(nn.Module):
    """Encoder step: MaxPool2d -> DoubleConv."""

    def __init__(self, in_channels: int, out_channels: int, dropout: float = 0.0):
        super().__init__()
        self.net = nn.Sequential(
            nn.MaxPool2d(2),
            DoubleConv(in_channels, out_channels, dropout=dropout),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)

In [12]:
class Up(nn.Module):
    """Decoder step: bilinear upsample -> concat skip -> DoubleConv."""

    def __init__(self, in_channels: int, out_channels: int, dropout: float = 0.0):
        super().__init__()
        self.up = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False)
        # After concat with skip: in_channels + in_channels/2
        self.conv = DoubleConv(in_channels + in_channels // 2, out_channels, dropout=dropout)

    def forward(self, x: torch.Tensor, skip: torch.Tensor) -> torch.Tensor:
        x = self.up(x)
        # Pad if spatial dimensions don't match (can happen with odd input sizes)
        if x.shape != skip.shape:
            diff_h = skip.shape[2] - x.shape[2]
            diff_w = skip.shape[3] - x.shape[3]
            x = torch.nn.functional.pad(x, (diff_w // 2, diff_w - diff_w // 2, diff_h // 2, diff_h - diff_h // 2))
        x = torch.cat([x, skip], dim=1)
        x = self.conv(x)
        return x

In [13]:
class UNet(nn.Module):
    """U-Net encoder-decoder with skip connections for binary segmentation."""

    def __init__(
        self,
        in_channels: int = 3,
        num_classes: int = 2,
        depth: int = 4,
        base_channels: int = 64,
        dropout: float = 0.0,
    ):
        super().__init__()
        self.in_channels = in_channels
        self.num_classes = num_classes
        self.depth = depth
        self.base_channels = base_channels

        self.enc0 = DoubleConv(in_channels, base_channels, dropout=dropout)
        self.down = nn.ModuleList()
        for i in range(depth):
            in_ch = base_channels * (2 ** i)
            out_ch = base_channels * (2 ** (i + 1))
            self.down.append(Down(in_ch, out_ch, dropout=dropout))

        bottleneck_ch = base_channels * (2 ** depth)
        self.bottleneck = DoubleConv(bottleneck_ch, bottleneck_ch, dropout=dropout)

        self.up = nn.ModuleList()
        for i in range(depth, 0, -1):
            in_ch = base_channels * (2 ** i)
            out_ch = base_channels * (2 ** (i - 1))
            self.up.append(Up(in_ch, out_ch, dropout=dropout))

        self.final_conv = nn.Conv2d(base_channels, num_classes, kernel_size=1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Encoder -> bottleneck -> decoder with skip connections -> logits [B, C, H, W]."""
        skips = []
        out = self.enc0(x)
        skips.append(out)

        for down_block in self.down:
            out = down_block(out)
            skips.append(out)

        out = self.bottleneck(out)

        for i, up_block in enumerate(self.up):
            skip = skips[-(i + 2)]
            out = up_block(out, skip)

        logits = self.final_conv(out)
        return logits

In [14]:
# Test U-Net instantiation with binary classes
model = UNet(in_channels=3, num_classes=2, depth=4, base_channels=64, dropout=0.1)
model = model.to(device)
print(f"U-Net created. Parameters: {sum(p.numel() for p in model.parameters()):,}")

# Test forward pass
x = torch.randn(1, 3, 1024, 1024, device=device)
with torch.no_grad():
    out = model(x)
print(f"Input shape: {x.shape}, Output shape: {out.shape}")
assert out.shape == (1, 2, 1024, 1024), f"Expected (1, 2, 1024, 1024), got {out.shape}"
print("✓ U-Net forward pass OK")

U-Net created. Parameters: 50,263,362
Input shape: torch.Size([1, 3, 1024, 1024]), Output shape: torch.Size([1, 2, 1024, 1024])
✓ U-Net forward pass OK


## Part 4: Training Pipeline

Complete training system with:
- Dynamic DataLoader worker calculation (based on available RAM)
- Binary Segmentation Metrics (Accuracy, Precision, Recall, F1, ROC-AUC, Dice)
- Augmentation pipelines (train-specific)
- Optimizer: AdamW with weight decay
- LR Scheduler: Linear warmup (5 epochs) -> Cosine annealing
- Checkpointing: saves best model (by Val F1 or Accuracy) + last model
- TensorBoard logging

### Training Workflow

1. Load data (train/val/test splits grouped by biopsia)
2. Create model, optimizer, scheduler
3. Train for N epochs:
   - Forward pass on batch
   - Compute loss (CrossEntropyLoss + Dice with class weights for binary classification)
   - Backward pass + gradient clipping
   - Optimizer step
   - Validate after each epoch (compute binary metrics on val set)
   - Save checkpoint if best model found
4. Final evaluation on test set
5. Save metrics report

In [15]:
def compute_dataloader_workers(batch_size: int = 8) -> int:
    """Compute optimal number of DataLoader workers based on available RAM."""
    try:
        import psutil
        available_gb = psutil.virtual_memory().available / (1024 ** 3)
    except ImportError:
        available_gb = 2.0

    # Float32 RGB tile: (1024*1024*3*4) bytes = ~12MB
    tile_gb = (1024 * 1024 * 3 * 4) / (1024 ** 3)

    # Each worker holds ~batch_size tiles × 2 (current + prefetch)
    # Use 50% of available RAM (conservative with 28 GB available)
    workers = int((available_gb * 0.5) / (tile_gb * batch_size * 2))
    workers = max(2, min(workers, os.cpu_count() or 4))

    return workers

print(f"Optimal DataLoader workers: {compute_dataloader_workers()}")


Optimal DataLoader workers: 4


In [16]:
class MeanIoUMetric:
    """Intersection over Union for per-class segmentation evaluation."""

    def __init__(self, num_classes: int, ignore_background: bool = False):
        self.num_classes = num_classes
        self.ignore_background = ignore_background
        self.reset()

    def reset(self):
        self.intersection = np.zeros(self.num_classes)
        self.union = np.zeros(self.num_classes)

    def update(self, pred: torch.Tensor, target: torch.Tensor):
        """Accumulate intersection and union per class from batch."""
        pred_labels = torch.argmax(pred, dim=1)

        pred_labels = pred_labels.cpu().numpy()
        target = target.cpu().numpy()

        for c in range(self.num_classes):
            pred_c = (pred_labels == c)
            target_c = (target == c)

            self.intersection[c] += np.logical_and(pred_c, target_c).sum()
            self.union[c] += np.logical_or(pred_c, target_c).sum()

    def compute(self) -> Tuple[float, dict]:
        """Return mean IoU and per-class dict."""
        per_class_iou = {}
        iou_scores = []

        for c in range(self.num_classes):
            if self.union[c] == 0:
                iou = 0.0
            else:
                iou = self.intersection[c] / self.union[c]

            if not (self.ignore_background and c == 0):
                iou_scores.append(iou)

            per_class_iou[f'class_{c}'] = float(iou)

        mean_iou = float(np.mean(iou_scores)) if iou_scores else 0.0
        return mean_iou, per_class_iou

In [17]:
class BinarySegmentationMetrics:
    """Accuracy, Precision, Recall, F1, Dice, and ROC-AUC for binary segmentation."""

    def __init__(self, max_auc_pixels: int = 100000):
        """Store subsampled pixels to prevent memory explosion with large images."""
        self.max_auc_pixels = max_auc_pixels
        self.reset()

    def reset(self):
        self.tp = 0
        self.tn = 0
        self.fp = 0
        self.fn = 0
        
        self.all_probs = []
        self.all_targets = []
        self.total_auc_pixels = 0

    def update(self, pred: torch.Tensor, target: torch.Tensor):
        """Accumulate confusion matrix and subsampled pixel predictions."""
        probs = torch.nn.functional.softmax(pred, dim=1)
        
        prob_pos = probs[:, 1, :, :]
        pred_labels = torch.argmax(pred, dim=1)

        pred_labels = pred_labels.cpu().numpy()
        target = target.cpu().numpy()
        prob_pos = prob_pos.cpu().numpy()

        pred_flat = pred_labels.flatten()
        target_flat = target.flatten()
        prob_flat = prob_pos.flatten()

        self.tp += np.logical_and(target_flat == 1, pred_flat == 1).sum()
        self.tn += np.logical_and(target_flat == 0, pred_flat == 0).sum()
        self.fp += np.logical_and(target_flat == 0, pred_flat == 1).sum()
        self.fn += np.logical_and(target_flat == 1, pred_flat == 0).sum()

        budget = self.max_auc_pixels - len(self.all_probs)
        if budget > 0:
            if len(prob_flat) <= budget:
                self.all_probs.extend(prob_flat)
                self.all_targets.extend(target_flat)
            else:
                idx = np.random.choice(len(prob_flat), size=budget, replace=False)
                self.all_probs.extend(prob_flat[idx])
                self.all_targets.extend(target_flat[idx])
        
        self.total_auc_pixels += len(prob_flat)

    def compute(self) -> dict:
        """Return accuracy, precision, recall, f1, dice, roc_auc."""
        metrics = {}

        total = self.tp + self.tn + self.fp + self.fn
        if total > 0:
            metrics['accuracy'] = float((self.tp + self.tn) / total)
        else:
            metrics['accuracy'] = 0.0

        if (self.tp + self.fp) > 0:
            metrics['precision'] = float(self.tp / (self.tp + self.fp))
        else:
            metrics['precision'] = 0.0

        if (self.tp + self.fn) > 0:
            metrics['recall'] = float(self.tp / (self.tp + self.fn))
        else:
            metrics['recall'] = 0.0

        if (metrics['precision'] + metrics['recall']) > 0:
            metrics['f1'] = float(
                2 * (metrics['precision'] * metrics['recall']) / 
                (metrics['precision'] + metrics['recall'])
            )
        else:
            metrics['f1'] = 0.0

        if (2 * self.tp + self.fp + self.fn) > 0:
            metrics['dice_metric'] = float(
                (2 * self.tp) / (2 * self.tp + self.fp + self.fn + 1e-8)
            )
        else:
            metrics['dice_metric'] = 0.0

        if len(self.all_targets) > 0 and len(set(self.all_targets)) > 1:
            try:
                from sklearn.metrics import roc_auc_score
                auc = roc_auc_score(self.all_targets, self.all_probs)
                metrics['roc_auc'] = float(auc)
            except Exception as e:
                print(f"Warning: Could not compute ROC-AUC: {e}")
                metrics['roc_auc'] = 0.0
        else:
            metrics['roc_auc'] = 0.0

        return metrics

In [ ]:
def get_transforms(reinhard_norm=None, channel_means=None, channel_stds=None, size: int = 1024):
    """Return augmentation pipelines for train/val (Reinhard and Z-score applied in-dataset)."""
    train_augment = A.Compose([
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.RandomRotate90(p=0.75),
        A.Transpose(p=0.5),
        A.Rotate(limit=15, p=0.3),
        A.GaussNoise(mean=0, std=(0.01, 0.05), p=0.2),
        A.GaussianBlur(blur_limit=3, p=0.2),
    ], additional_targets={'mask': 'mask'})
    
    train_transform = train_augment
    val_transform = A.Compose([])
    
    return train_transform, val_transform

In [19]:
def train_epoch(
    model: nn.Module,
    dataloader,
    criterion: nn.Module,
    optimizer: optim.Optimizer,
    device: torch.device,
    scaler: GradScaler = None,
) -> float:
    """Train one epoch with optional AMP (FP16) and gradient clipping."""
    model.train()
    total_loss = 0.0
    num_batches = 0

    for images, masks in dataloader:
        images = images.to(device)
        masks = masks.to(device)

        optimizer.zero_grad(set_to_none=True)

        if scaler is not None:
            with autocast():
                logits = model(images)
                loss = criterion(logits, masks)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
        else:
            logits = model(images)
            loss = criterion(logits, masks)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

        total_loss += loss.item()
        num_batches += 1

    avg_loss = total_loss / num_batches if num_batches > 0 else 0.0
    return avg_loss

In [20]:
def eval_epoch(
    model: nn.Module,
    dataloader,
    criterion: nn.Module,
    device: torch.device,
) -> Tuple[float, dict]:
    """Evaluate one epoch; return loss and binary segmentation metrics."""
    model.eval()
    total_loss = 0.0
    num_batches = 0
    seg_metrics = BinarySegmentationMetrics()
    iou_metric = MeanIoUMetric(num_classes=2)

    with torch.no_grad():
        for images, masks in dataloader:
            images = images.to(device)
            masks = masks.to(device)

            with autocast():
                logits = model(images)
                loss = criterion(logits, masks)

            total_loss += loss.item()
            num_batches += 1

            seg_metrics.update(logits, masks)
            iou_metric.update(logits, masks)

    avg_loss = total_loss / num_batches if num_batches > 0 else 0.0
    metrics = seg_metrics.compute()
    mean_iou, per_class_iou = iou_metric.compute()
    metrics['mean_iou'] = mean_iou
    metrics.update(per_class_iou)

    return avg_loss, metrics

In [21]:
def train_unet(
    epochs: int = 50,
    batch_size: int = 8,
    lr: float = 1e-3,
    images_dir: str = 'Salidas/Tiles_UNet',
    output_dir: str = 'checkpoints',
    num_workers: int = None,
    seed: int = 42,
    warmup_epochs: int = 5,
    use_amp: bool = True,
):
    """Complete training pipeline: data loading, model training, validation, checkpointing."""
    torch.manual_seed(seed)
    np.random.seed(seed)

    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    run_name = f"unet_binary_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
    log_dir = output_dir / run_name
    writer = SummaryWriter(str(log_dir))

    print(f"Run: {run_name}")
    print(f"Device: {device}")
    print(f"AMP (FP16): {use_amp and device.type == 'cuda'}")
    print(f"Output dir: {output_dir}")

    print("\n--- Data Loading Pipeline ---")
    print("Step 1: Computing grouped split by biopsia...")
    train_biopsias, val_biopsias, test_biopsias, biopsias_dict = split_biopsias(
        images_dir=images_dir,
        train_size=0.70,
        val_size=0.15,
        seed=seed,
    )
    print(f"  Train biopsies: {len(train_biopsias)}")
    print(f"  Val biopsies: {len(val_biopsias)}")
    print(f"  Test biopsies: {len(test_biopsias)}")

    print("\nStep 2: Collecting train image paths...")
    train_image_paths = []
    for biopsia in train_biopsias:
        train_image_paths.extend(biopsias_dict[biopsia])
    print(f"  Train tiles: {len(train_image_paths)}")

    print("\nStep 3: Computing Reinhard stain normalization template...")
    try:
        reinhard_stats = ReinhardNormalize.compute_template_stats(train_image_paths, n_samples=200)
        reinhard_norm = ReinhardNormalize(reinhard_stats)
        print(f"  Template stats computed from ~{min(200, len(train_image_paths))} samples")
        print(f"    L: mean={reinhard_stats['mean_L']:.1f}, std={reinhard_stats['std_L']:.1f}")
        print(f"    a: mean={reinhard_stats['mean_a']:.1f}, std={reinhard_stats['std_a']:.1f}")
        print(f"    b: mean={reinhard_stats['mean_b']:.1f}, std={reinhard_stats['std_b']:.1f}")
    except Exception as e:
        print(f"  Warning: Could not compute Reinhard stats: {e}")
        reinhard_norm = None

    print("\nStep 4: Computing per-channel Z-score normalization stats...")
    try:
        channel_means, channel_stds = compute_channel_stats(train_image_paths, n_samples=200)
        print(f"  Channel means (RGB): {[f'{m:.4f}' for m in channel_means]}")
        print(f"  Channel stds (RGB):  {[f'{s:.4f}' for s in channel_stds]}")
    except Exception as e:
        print(f"  Warning: Could not compute channel stats: {e}")
        channel_means = None
        channel_stds = None

    print("\nStep 5: Creating augmentation pipelines...")
    train_transforms, val_transforms = get_transforms(
        reinhard_norm=reinhard_norm,
        channel_means=channel_means,
        channel_stds=channel_stds,
    )

    if num_workers is None:
        num_workers = compute_dataloader_workers(batch_size=batch_size)
        print(f"  Auto-calculated DataLoader workers: {num_workers}")
    else:
        print(f"  Using specified DataLoader workers: {num_workers}")

    print("\nStep 5b: Creating dataloaders with pre-computed splits and normalization...")
    train_loader, val_loader, test_loader = create_dataloaders(
        images_dir=images_dir,
        batch_size=batch_size,
        num_workers=num_workers,
        seed=seed,
        train_transforms=train_transforms,
        val_transforms=val_transforms,
        reinhard_norm=reinhard_norm,
        channel_means=channel_means,
        channel_stds=channel_stds,
    )

    print(f"  Train: {len(train_loader.dataset)} tiles | "
          f"Val: {len(val_loader.dataset)} tiles | "
          f"Test: {len(test_loader.dataset)} tiles")

    print("\nCreating model...")
    model = UNet(in_channels=3, num_classes=2, depth=4, base_channels=64, dropout=0.1)
    model = model.to(device)
    print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

    print("\nComputing class weights from training set...")
    try:
        class_weights = compute_class_weights(train_loader.dataset.mask_paths, num_classes=2)
        print(f"Class weights (background, glomerulus): {class_weights.tolist()}")
        class_weights = class_weights.to(device)
    except Exception as e:
        print(f"Warning: Could not compute class weights: {e}. Using uniform weights.")
        class_weights = None

    criterion = CombinedLoss(num_classes=2, weight_ce=0.5, weight_dice=0.5,
                             class_weights=class_weights)
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)

    scaler = GradScaler() if (use_amp and device.type == 'cuda') else None
    if scaler:
        print("AMP (FP16) enabled — GradScaler initialized")

    scheduler = torch.optim.lr_scheduler.SequentialLR(
        optimizer,
        schedulers=[
            LinearLR(optimizer, start_factor=0.1, end_factor=1.0, total_iters=warmup_epochs),
            CosineAnnealingLR(optimizer, T_max=epochs - warmup_epochs, eta_min=1e-6),
        ],
        milestones=[warmup_epochs],
    )

    threshold = 0.5
    best_val_f1 = 0.0
    train_history = []

    print("\nStarting training...")
    for epoch in range(epochs):
        print(f"\n{'='*60}")
        print(f"Epoch [{epoch+1}/{epochs}]")
        print(f"{'='*60}")

        train_loss = train_epoch(model, train_loader, criterion, optimizer, device, scaler=scaler)
        print(f"Train Loss: {train_loss:.4f}")

        val_loss, val_metrics = eval_epoch(model, val_loader, criterion, device)
        print(f"Val Loss: {val_loss:.4f}")
        print(f"Val Metrics:")
        print(f"  Accuracy:  {val_metrics['accuracy']:.4f}")
        print(f"  Precision: {val_metrics['precision']:.4f}")
        print(f"  Recall:    {val_metrics['recall']:.4f}")
        print(f"  F1:        {val_metrics['f1']:.4f}")
        print(f"  Dice:      {val_metrics.get('dice_metric', 0.0):.4f}")
        print(f"  mIoU:      {val_metrics.get('mean_iou', 0.0):.4f}")
        print(f"  ROC-AUC:   {val_metrics['roc_auc']:.4f}")

        scheduler.step()
        current_lr = optimizer.param_groups[0]['lr']
        print(f"LR: {current_lr:.2e}")

        checkpoint = {
            'epoch': epoch + 1,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'scaler_state_dict': scaler.state_dict() if scaler else None,
            'best_val_f1': best_val_f1,
            'class_weights': class_weights.cpu().tolist() if class_weights is not None else None,
            'channel_means': channel_means,
            'channel_stds': channel_stds,
            'reinhard_stats': reinhard_stats if reinhard_norm else None,
            'model_config': {
                'in_channels': 3,
                'num_classes': 2,
                'depth': 4,
                'base_channels': 64,
                'dropout': 0.1,
            },
            'threshold': threshold,
        }

        torch.save(checkpoint, output_dir / f'{run_name}_last.pth')

        if val_metrics['f1'] > best_val_f1:
            best_val_f1 = val_metrics['f1']
            checkpoint['best_val_f1'] = best_val_f1
            torch.save(checkpoint, output_dir / f'{run_name}_best.pth')
            print(f"✓ Best model saved! (Val F1: {val_metrics['f1']:.4f})")

        writer.add_scalar('loss/train', train_loss, epoch)
        writer.add_scalar('loss/val', val_loss, epoch)
        writer.add_scalar('metric/val_accuracy', val_metrics['accuracy'], epoch)
        writer.add_scalar('metric/val_precision', val_metrics['precision'], epoch)
        writer.add_scalar('metric/val_recall', val_metrics['recall'], epoch)
        writer.add_scalar('metric/val_f1', val_metrics['f1'], epoch)
        writer.add_scalar('metric/val_dice_metric', val_metrics.get('dice_metric', 0.0), epoch)
        writer.add_scalar('metric/val_mean_iou', val_metrics.get('mean_iou', 0.0), epoch)
        writer.add_scalar('metric/val_roc_auc', val_metrics['roc_auc'], epoch)
        writer.add_scalar('lr', current_lr, epoch)

        train_history.append({
            'epoch': epoch + 1,
            'train_loss': train_loss,
            'val_loss': val_loss,
            'val_metrics': val_metrics,
        })

    writer.close()

    print(f"\n{'='*60}")
    print("Final evaluation on test set...")
    print(f"{'='*60}")

    best_ckpt = torch.load(output_dir / f'{run_name}_best.pth', map_location=device)
    model.load_state_dict(best_ckpt['model_state_dict'])

    test_loss, test_metrics = eval_epoch(model, test_loader, criterion, device)
    print(f"Test Loss: {test_loss:.4f}")
    print(f"Test Metrics:")
    print(f"  Accuracy:  {test_metrics['accuracy']:.4f}")
    print(f"  Precision: {test_metrics['precision']:.4f}")
    print(f"  Recall:    {test_metrics['recall']:.4f}")
    print(f"  F1:        {test_metrics['f1']:.4f}")
    print(f"  Dice:      {test_metrics.get('dice_metric', 0.0):.4f}")
    print(f"  mIoU:      {test_metrics.get('mean_iou', 0.0):.4f}")
    print(f"  ROC-AUC:   {test_metrics['roc_auc']:.4f}")

    report = {
        'run_name': run_name,
        'model_config': {
            'in_channels': 3,
            'num_classes': 2,
            'depth': 4,
            'base_channels': 64,
            'dropout': 0.1,
        },
        'training_config': {
            'epochs': epochs,
            'batch_size': batch_size,
            'learning_rate': lr,
            'warmup_epochs': warmup_epochs,
            'use_amp': use_amp,
        },
        'normalization_params': {
            'reinhard_stats': reinhard_stats if reinhard_norm else None,
            'channel_means': channel_means,
            'channel_stds': channel_stds,
            'class_weights': class_weights.cpu().tolist() if class_weights is not None else None,
            'threshold': threshold,
        },
        'final_metrics': {
            'train_loss': train_loss,
            'val_loss': val_loss,
            'val_metrics': val_metrics,
            'test_loss': test_loss,
            'test_metrics': test_metrics,
        },
        'training_history': train_history,
    }

    with open(output_dir / f'{run_name}_report.json', 'w') as f:
        json.dump(report, f, indent=2)

    print(f"\n✓ Training complete!")
    print(f"Checkpoints saved to: {output_dir}")
    print(f"TensorBoard logs: tensorboard --logdir {log_dir}")
    
    return model, report

## Part 6: Usage Instructions

### Running Training

Uncomment and run the cell below to start training:

In [ ]:
if Path('Salidas/Tiles_UNet').exists():
    model, report = train_unet(
        epochs=50,
        batch_size=8,          # Optimized for T4 15.93 GB VRAM with AMP
        lr=1e-3,
        images_dir='Salidas/Tiles_UNet',
        output_dir='checkpoints',
        num_workers=None,      # Auto-calculate based on 28 GB RAM
        seed=42,
        warmup_epochs=5,
        use_amp=True,          # AMP (FP16) for T4 Tensor Cores — 2–3× speedup
    )
else:
    print("Run the preprocessing pipeline first:")
    print("  python tiling_unet.py")

## Part 7: Troubleshooting and System Requirements

### Troubleshooting

**"No images found in Salidas/Estandarizados"**
- Run the preprocessing pipeline first (tiling -> normalization -> standardization).

**"CUDA out of memory"**
- Reduce `batch_size` (try 2 or 1)
- Increase `--ram-fraction` in preprocessing scripts
- Use CPU training (slower but uses less VRAM)

**"Expected masks directory"**
- Ensure masks were copied by normalizacion.py and estandarizacion.py to `Salidas/Estandarizados/*/masks/`
- Verify mask naming convention: `{tile_name}_mask.png`

**"No valid image-mask pairs found"**
- Check that image and mask counts match
- Verify paths: images in `*/images/` and masks in `*/masks/`

### System Requirements

- **Python**: 3.10+
- **PyTorch**: 2.0+ (with CUDA if GPU available)
- **RAM**: 16GB minimum, 32GB+ recommended
- **GPU**: Optional but recommended (4GB VRAM minimum)
- **Dependencies**: See requirements.txt (numpy, torch, torchvision, opencv, scikit-learn, albumentations, tensorboard)

### References

- **U-Net**: Ronneberger et al., "U-Net: Convolutional Networks for Biomedical Image Segmentation" (MICCAI 2015)
- **Dice Loss**: Sørensen–Dice coefficient for segmentation evaluation
- **Class Weighting**: Inverse frequency weighting for class imbalance handling

In [ ]:
def visualize_predictions(model, dataset, n_samples: int = 8, threshold: float = 0.5, device=None):
    """Visualize predictions vs ground truth on random dataset samples."""
    if device is None:
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    
    import matplotlib.pyplot as plt
    import matplotlib.patches as mpatches
    from matplotlib.colors import ListedColormap
    
    model.eval()
    
    sample_indices = np.random.choice(len(dataset), size=min(n_samples, len(dataset)), replace=False)
    
    n_cols = 3
    fig, axes = plt.subplots(len(sample_indices), n_cols, figsize=(15, 5 * len(sample_indices)))
    
    if len(sample_indices) == 1:
        axes = axes.reshape(1, -1)
    
    with torch.no_grad():
        for row, idx in enumerate(sample_indices):
            img_tensor, mask_tensor = dataset[idx]
            img_tensor = img_tensor.unsqueeze(0).to(device)
            mask_tensor = mask_tensor.cpu().numpy()
            
            logits = model(img_tensor)
            probs = torch.softmax(logits, dim=1)
            pred_probs = probs[0, 1, :, :].cpu().numpy()
            pred_mask = (pred_probs > threshold).astype(np.uint8)
            
            img_np = img_tensor[0].permute(1, 2, 0).cpu().numpy()
            if dataset.channel_means is not None and dataset.channel_stds is not None:
                for c in range(3):
                    img_np[..., c] = img_np[..., c] * dataset.channel_stds[c] + dataset.channel_means[c]
            img_np = np.clip(img_np, 0, 1)
            
            axes[row, 0].imshow(img_np)
            axes[row, 0].set_title(f'Input Image (Sample {idx})')
            axes[row, 0].axis('off')
            
            axes[row, 1].imshow(img_np, alpha=0.7)
            axes[row, 1].imshow(mask_tensor, cmap='Greens', alpha=0.4, vmin=0, vmax=1)
            axes[row, 1].set_title(f'Ground Truth Mask')
            axes[row, 1].axis('off')
            
            axes[row, 2].imshow(img_np, alpha=0.7)
            axes[row, 2].imshow(pred_mask, cmap='Reds', alpha=0.4, vmin=0, vmax=1)
            axes[row, 2].set_title(f'Predicted Mask (Dice: {2*((pred_mask & mask_tensor).sum())/(2*(pred_mask | mask_tensor).sum() + 1e-8):.3f})')
            axes[row, 2].axis('off')
    
    plt.tight_layout()
    plt.show()

## Part 8: Qualitative Evaluation — Predictions vs Ground Truth

Visual inspection of model predictions on the test set is critical in medical image segmentation. Quantitative metrics like F1 and accuracy can mask clinically relevant errors such as:
- Imprecise borders around glomeruli (false positives/negatives at edges)
- False positives in glomerulus-like structures (cracks, debris)
- Missed glomeruli in hard-to-segment regions

The cells below visualize predictions alongside ground truth for qualitative assessment.